In [29]:
import numpy as np
import pandas as pd
from pathlib import Path

## Data frame preview

In [30]:
titles_table = pd.read_csv("./archive/netflix_titles.csv")
titles_table.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


### Columns and rows

In [31]:
len(titles_table.columns)


12

In [32]:
titles_table.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='str')

In [33]:
len(titles_table.index)

8807

In [34]:
titles_table.index

RangeIndex(start=0, stop=8807, step=1)

In [35]:
titles_table.iloc[0]

show_id                                                        s1
type                                                        Movie
title                                        Dick Johnson Is Dead
director                                          Kirsten Johnson
cast                                                          NaN
country                                             United States
date_added                                     September 25, 2021
release_year                                                 2020
rating                                                      PG-13
duration                                                   90 min
listed_in                                           Documentaries
description     As her father nears the end of his life, filmm...
Name: 0, dtype: object

## Data quality analysis

In [36]:
data_quality_analysis = pd.DataFrame(
    columns=titles_table.columns, 
    index=["data type", "missing_values", "duplicates", "invalid values", ]
)
data_quality_analysis

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
data type,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
missing_values,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duplicates,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
invalid values,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
data_quality_analysis.loc["data type"] = titles_table.dtypes

data_quality_analysis.loc["missing_values"] = titles_table.isna().astype(int).sum()

data_quality_analysis.loc["duplicates"] = titles_table.apply(lambda col: col.duplicated(), axis="index").astype(int).sum()

duration_pattern = r"\d+\s+(?:min|Seasons?)"
invalid_duration_mask = titles_table["rating"].str.match(duration_pattern)
data_quality_analysis.at["invalid values", "rating"] = len(titles_table[invalid_duration_mask])

# invalid values were calculated only for rating column as it was already known it has data entry error (values from duration column)
data_quality_analysis

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
data type,str,str,str,str,str,str,str,int64,str,str,str,str
missing_values,0,0,0,2634,825,831,10,0,4,3,0,0
duplicates,0,8805,0,4278,1114,8058,7039,8733,8789,8586,8293,32
invalid values,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN


In [65]:
titles_table[invalid_duration_mask]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
5541,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,"April 4, 2017",2017,74 min,NaN,Movies,"Louis C.K. muses on religion, eternal love, gi..."
5794,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,"September 16, 2016",2010,84 min,NaN,Movies,Emmy-winning comedy writer Louis C.K. brings h...
5813,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,"August 15, 2016",2015,66 min,NaN,Movies,The comic puts his trademark hilarious/thought...


## Data format comparison

Comparing feather and csv format

### Writing time

In [51]:
%%timeit
titles_table.to_feather("./archive/netflix_titles_copy.feather")

4.17 ms ± 43.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [50]:
%%timeit
titles_table.to_csv("./archive/netflix_titles_copy.csv")

42.4 ms ± 1.35 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### Reading time

In [40]:
%%timeit
pd.read_feather("./archive/netflix_titles.feather")

1.25 ms ± 27.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [58]:
feather_format = pd.read_feather("./archive/netflix_titles.feather")

In [41]:
%%timeit
pd.read_csv("./archive/netflix_titles.csv")

29.7 ms ± 38.8 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [59]:
csv_format = pd.read_csv("./archive/netflix_titles.csv")

### File size

In [42]:
f"{Path("./archive/netflix_titles.feather").stat().st_size} bytes"

'2423218 bytes'

In [43]:
f"{Path("./archive/netflix_titles.csv").stat().st_size} bytes"

'3399671 bytes'

### Preservation of rows and columns, data types, and missing values

In [60]:
feather_format.equals(csv_format)

True